In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# Your OANDA API Key
api_key = "6a591467afc13858de5819fcd0b71ea5-cd62b3b1abd7867ebef4904b30743701"

# Base URL
base_url = "https://api-fxpractice.oanda.com/v3/instruments"

# Currency pairs and their correct instrument names
currency_pairs = {
    "USD_CHF": "USD_CHF",
    "EUR_CHF": "EUR_CHF",
    "USD_EUR": "EUR_USD"
}

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# Define start date for historical data
start_date = datetime(2000, 1, 1)  # Change this to the desired starting year
end_date = datetime.now()

# Loop through currency pairs
for pair, instrument in currency_pairs.items():
    all_data = []
    current_start = start_date

    while current_start < end_date:
        current_end = current_start + timedelta(days=365)  # Fetch one year at a time
        if current_end > end_date:
            current_end = end_date
        
        url = f"{base_url}/{instrument}/candles"
        params = {
            "granularity": "M",  # Monthly data
            "price": "M",        # Midpoint prices
            "from": current_start.strftime('%Y-%m-%dT%H:%M:%SZ'),
            "to": current_end.strftime('%Y-%m-%dT%H:%M:%SZ')
        }
        
        try:
            response = requests.get(url, headers=headers, params=params)
            response.raise_for_status()
            
            data = response.json()
            candles = data.get("candles", [])
            
            if candles:
                # Convert to DataFrame
                df = pd.DataFrame(candles)
                
                # Convert time to datetime and format it
                df["time"] = pd.to_datetime(df["time"]).dt.strftime('%Y-%m-%d')
                
                # Extract only the closing price
                df['rate'] = df['mid'].apply(lambda x: float(x['c']))
                
                # Keep only the columns we need
                df = df[['time', 'rate']]
                
                # Rename columns
                df.columns = ['Date', 'Rate']
                
                all_data.append(df)
            
            print(f"Fetched data for {pair} from {current_start} to {current_end}")
        
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data for {pair} from {current_start} to {current_end}: {str(e)}")
        
        # Move to the next range
        current_start = current_end

    # Combine all data into a single DataFrame
    if all_data:
        full_data = pd.concat(all_data, ignore_index=True)
        full_data.set_index("Date", inplace=True)
        
        # Save to CSV
        filename = f"{pair}_historical_monthly_rates.csv"
        full_data.to_csv(filename)
        print(f"Historical data for {pair} saved to {filename}")
        print(full_data.head())


Fetched data for USD_CHF from 2000-01-01 00:00:00 to 2000-12-31 00:00:00
Fetched data for USD_CHF from 2000-12-31 00:00:00 to 2001-12-31 00:00:00
Fetched data for USD_CHF from 2001-12-31 00:00:00 to 2002-12-31 00:00:00
Fetched data for USD_CHF from 2002-12-31 00:00:00 to 2003-12-31 00:00:00
Fetched data for USD_CHF from 2003-12-31 00:00:00 to 2004-12-30 00:00:00
Fetched data for USD_CHF from 2004-12-30 00:00:00 to 2005-12-30 00:00:00
Fetched data for USD_CHF from 2005-12-30 00:00:00 to 2006-12-30 00:00:00
Fetched data for USD_CHF from 2006-12-30 00:00:00 to 2007-12-30 00:00:00
Fetched data for USD_CHF from 2007-12-30 00:00:00 to 2008-12-29 00:00:00
Fetched data for USD_CHF from 2008-12-29 00:00:00 to 2009-12-29 00:00:00
Fetched data for USD_CHF from 2009-12-29 00:00:00 to 2010-12-29 00:00:00
Fetched data for USD_CHF from 2010-12-29 00:00:00 to 2011-12-29 00:00:00
Fetched data for USD_CHF from 2011-12-29 00:00:00 to 2012-12-28 00:00:00
Fetched data for USD_CHF from 2012-12-28 00:00:00 t

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b0d79e99-778c-4964-b163-e34d369ad413' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>